# CONCODE Neural Brain — Google Colab Training Notebook

Train all 3 models on **Google Colab free T4 GPU**:

| Model | Base | Purpose | Params |
|-------|------|---------|--------|
| **CodeLlama-7B** | `unsloth/codellama-7b-instruct-bnb-4bit` | Multi-guild code generation | **7 Billion** |
| **DistilBERT** | `distilbert-base-uncased` | Intent classification (11 guilds) | 66M |
| **CodeT5-small** | `Salesforce/codet5-small` | Code syntax repair | 60M |

### Instructions
1. Set runtime to **GPU (T4)**: Runtime → Change runtime type → T4 GPU
2. Run cells **in order** (1 through 7)
3. Upload your data files when prompted (Cell 2)
4. Download trained models at the end (Cell 7)

**Total training time: ~45-60 minutes on free T4**

In [ ]:
# ════════════════════════════════════════════════════════════
# Cell 1: Install All Dependencies
# ════════════════════════════════════════════════════════════

!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install datasets transformers[torch] scikit-learn sentencepiece
!pip install protobuf

print("\n[OK] All dependencies installed!")

# Verify GPU
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ════════════════════════════════════════════════════════════
# Cell 2: Upload Training Data
# ════════════════════════════════════════════════════════════

import os

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REQUIRED_FILES = [
    'multi_guild_train.jsonl',
    'intent_dataset.jsonl',
    'repair_dataset.jsonl'
]

if IN_COLAB:
    print("Please upload these 3 data files:")
    for f in REQUIRED_FILES:
        print(f"   - {f}")
    print()
    uploaded = files.upload()
    for fn in uploaded.keys():
        print(f"  [OK] Uploaded: {fn} ({len(uploaded[fn]):,} bytes)")
else:
    print("Running locally — checking for files in current directory")

# Verify all files present
print("\nVerifying data files...")
for f in REQUIRED_FILES:
    if os.path.exists(f):
        size = os.path.getsize(f)
        lines = sum(1 for _ in open(f))
        print(f"  [OK] {f}: {lines:,} examples ({size:,} bytes)")
    else:
        print(f"  [FAIL] {f}: NOT FOUND")

In [ ]:
# ════════════════════════════════════════════════════════════
# Cell 3: Fine-tune CodeLlama-7B with QLoRA
# ════════════════════════════════════════════════════════════
# Architecture:
#   - 7 Billion parameters
#   - 32 transformer layers
#   - 32 attention heads
#   - 4096 hidden dimension
#   - LoRA r=64, alpha=128 for maximum adaptation
#   - Targets all projection layers (Q, K, V, O, gate, up, down)
# ════════════════════════════════════════════════════════════

from unsloth import FastLanguageModel
import torch
import json
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

print("Loading CodeLlama-7B-Instruct (4-bit quantized)...")
print("  • 7 Billion parameters")
print("  • 32 transformer layers")
print("  • 32 attention heads")
print("  • 4096 hidden dimension")
print()

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/codellama-7b-instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

print("\nAdding LoRA adapters (r=64, alpha=128)...")
print("  Targeting: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj")

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n  Total parameters:     {total:,}")
print(f"  Trainable parameters: {trainable:,}")
print(f"  Trainable ratio:      {trainable/total*100:.2f}%")

# Load multi-guild training data
print("\nLoading multi-guild training data...")
data = []
with open("multi_guild_train.jsonl") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

dataset = Dataset.from_list(data)
print(f"  Loaded {len(dataset)} training examples")

# Format prompts for CodeLlama instruct
def formatting_func(examples):
    texts = []
    for sys, inst, out in zip(examples["system"], examples["instruction"], examples["output"]):
        text = f"<s>[INST] <<SYS>>\n{sys}\n<</SYS>>\n\n{inst} [/INST] {out}</s>"
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_func, batched=True)

# Training
print("\n" + "="*60)
print("  Starting QLoRA training...")
print("="*60)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        max_steps=300,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        output_dir="outputs_codellama",
        optim="adamw_8bit",
        seed=42,
        save_steps=100,
    ),
)

trainer.train()
print("\n[OK] CodeLlama-7B QLoRA training complete!")

In [ ]:
# ════════════════════════════════════════════════════════════
# Cell 4: Merge LoRA → 16-bit & Save
# ════════════════════════════════════════════════════════════

print("Merging LoRA adapters into base model (16-bit)...")

merged_dir = "codellama-7b-concode-merged"

model.save_pretrained_merged(
    merged_dir,
    tokenizer,
    save_method="merged_16bit",
)

print(f"\n[OK] Merged model saved to: {merged_dir}/")

# Show directory contents
import os
files_list = os.listdir(merged_dir)
total_size = sum(os.path.getsize(os.path.join(merged_dir, f)) for f in files_list)
print(f"   Files: {len(files_list)}")
print(f"   Total size: {total_size / 1e9:.2f} GB")

# Clean up GPU memory
del model, tokenizer, trainer
import gc
gc.collect()
torch.cuda.empty_cache()
print("\n[OK] GPU memory cleared for next model")

In [ ]:
# ════════════════════════════════════════════════════════════
# Cell 5: Fine-tune DistilBERT for Intent Classification
# ════════════════════════════════════════════════════════════
# 11 guilds: debugger, webgl, ai_ml, qa, security, performance,
#            database, ui_ux, devops, api, fullstack
# ════════════════════════════════════════════════════════════

import json
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

print("Loading intent classification data...")
data = []
with open("intent_dataset.jsonl") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

# Encode labels
le = LabelEncoder()
labels = le.fit_transform([d["label"] for d in data])
label_names = list(le.classes_)
num_labels = len(label_names)

print(f"  Total examples: {len(data)}")
print(f"  Classes ({num_labels}): {label_names}")

# Create dataset
dataset = Dataset.from_dict({
    "text": [d["text"] for d in data],
    "label": labels.tolist(),
})

# Split 90/10
split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
eval_ds = split["test"]
print(f"  Train: {len(train_ds)}, Eval: {len(eval_ds)}")

# Load DistilBERT
print("\nLoading DistilBERT...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
)

# Set label mappings in model config
model.config.id2label = {i: name for i, name in enumerate(label_names)}
model.config.label2id = {name: i for i, name in enumerate(label_names)}

# Tokenize
def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

train_ds = train_ds.map(tokenize_fn, batched=True)
eval_ds = eval_ds.map(tokenize_fn, batched=True)
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
eval_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# Train
print("\n" + "="*60)
print("  Training DistilBERT Intent Classifier")
print("="*60)

training_args = TrainingArguments(
    output_dir="outputs_intent",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=5e-5,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

# Evaluate
results = trainer.evaluate()
print(f"\n[OK] Intent Classifier trained! Accuracy: {results['eval_accuracy']:.4f}")

# Save
intent_dir = "intent_classifier"
model.save_pretrained(intent_dir)
tokenizer.save_pretrained(intent_dir)
print(f"[OK] Saved to: {intent_dir}/")

# Cleanup
del model, tokenizer, trainer
import gc
gc.collect()
torch.cuda.empty_cache()
print("[OK] GPU memory cleared")

In [ ]:
# ════════════════════════════════════════════════════════════
# Cell 6: Fine-tune CodeT5-small for Code Repair
# ════════════════════════════════════════════════════════════

import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)

print("Loading code repair data...")
data = []
with open("repair_dataset.jsonl") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print(f"  Total pairs: {len(data)}")

dataset = Dataset.from_dict({
    "broken": [d["broken"] for d in data],
    "fixed": [d["fixed"] for d in data],
})

# Split 95/5
split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split["train"]
eval_ds = split["test"]
print(f"  Train: {len(train_ds)}, Eval: {len(eval_ds)}")

# Load CodeT5-small
print("\nLoading CodeT5-small...")
model_name = "Salesforce/codet5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Tokenize
MAX_INPUT_LEN = 512
MAX_OUTPUT_LEN = 512

def tokenize_fn(examples):
    inputs = [f"fix: {code}" for code in examples["broken"]]
    targets = examples["fixed"]
    model_inputs = tokenizer(
        inputs, max_length=MAX_INPUT_LEN, truncation=True, padding="max_length"
    )
    labels = tokenizer(
        targets, max_length=MAX_OUTPUT_LEN, truncation=True, padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["broken", "fixed"])
eval_ds = eval_ds.map(tokenize_fn, batched=True, remove_columns=["broken", "fixed"])

# Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Train
print("\n" + "="*60)
print("  Training CodeT5-small Code Repair Model")
print("="*60)

training_args = TrainingArguments(
    output_dir="outputs_repair",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=5e-5,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    seed=42,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
)

trainer.train()

# Save
repair_dir = "code_repair"
model.save_pretrained(repair_dir)
tokenizer.save_pretrained(repair_dir)
print(f"\n[OK] Code Repair model saved to: {repair_dir}/")

# Quick test
print("\nQuick test:")
test_input = "fix: functoin calculateTotal(data) { retun data.total; }"
inputs = tokenizer(test_input, return_tensors="pt", max_length=512, truncation=True).to(model.device)
outputs = model.generate(**inputs, max_length=512)
repaired = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"  Input:  {test_input}")
print(f"  Output: {repaired}")

# Cleanup
del model, tokenizer, trainer
import gc
gc.collect()
torch.cuda.empty_cache()
print("\n[OK] GPU memory cleared")

In [ ]:
# ════════════════════════════════════════════════════════════
# Cell 7: Archive & Download All Models
# ════════════════════════════════════════════════════════════

import shutil
import os

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

models_to_archive = [
    ("codellama-7b-concode-merged", "The main 7B LLM (convert to GGUF on Mac)"),
    ("intent_classifier", "DistilBERT intent classifier (11 guilds)"),
    ("code_repair", "CodeT5-small code repair model"),
]

print("Archiving trained models...\n")

for dir_name, description in models_to_archive:
    if os.path.isdir(dir_name):
        archive_name = f"{dir_name}.zip"
        print(f"  Archiving {dir_name}/ → {archive_name}")
        print(f"     ({description})")
        shutil.make_archive(dir_name, 'zip', '.', dir_name)
        size = os.path.getsize(archive_name)
        print(f"     Size: {size / 1e6:.1f} MB")
        print()
    else:
        print(f"  [WARNING] {dir_name}/ not found — skipping")

if IN_COLAB:
    print("\n" + "="*60)
    print("  Downloading models to your Mac...")
    print("  (Large files — this may take a few minutes)")
    print("="*60 + "\n")
    
    # Download smaller models first
    for dir_name, _ in models_to_archive:
        archive = f"{dir_name}.zip"
        if os.path.exists(archive):
            print(f"  ⬇️  Downloading {archive}...")
            files.download(archive)
else:
    print("\nNot in Colab — archives saved locally.")

print("\n" + "="*60)
print("  [OK] All models archived and ready!")
print()
print("  After downloading to your Mac:")
print("    1. Unzip intent_classifier.zip → models/intent_classifier/")
print("    2. Unzip code_repair.zip → models/code_repair/")
print("    3. Unzip codellama-7b-concode-merged.zip → models/codellama-7b-concode-merged/")
print("    4. Run: bash src/convert_to_gguf.sh")
print("    5. Start server: ./start.sh")
print("="*60)